In [1]:
import sys
from pathlib import Path

root = Path.cwd().resolve()
while root != root.parent and not (root / "shared").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from shared.paths import get_experiment_paths
from shared.visualization import show_grid, save_grid
from shared.seed import seed_everything
from shared.weight_init import dcgan_weights_init

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

seed_everything(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

paths = get_experiment_paths("07_wgan_gp_mnist")

In [2]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_ds = datasets.MNIST(
    root=str(paths.datasets_dir / "MNIST"),
    train=True,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=True
)

In [3]:
z_dim = 100
feat = 64

class WGAN_Generator(nn.Module):
    def __init__(self, z_dim=100, feat=64, img_channels=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, feat*4, 7, 1, 0, bias=False),
            nn.BatchNorm2d(feat*4),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*4, feat*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat*2),
            nn.ReLU(True),

            nn.ConvTranspose2d(feat*2, feat, 4, 2, 1, bias=False),
            nn.BatchNorm2d(feat),
            nn.ReLU(True),

            nn.Conv2d(feat, img_channels, 3, 1, 1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.net(z)

class WGAN_Critic(nn.Module):
    def __init__(self, feat=64, img_channels=1):
        super().__init__()
        # NOTE: no sigmoid at the end
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, feat, 4, 2, 1),            # 28 -> 14
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat, feat*2, 4, 2, 1),                  # 14 -> 7
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*2, feat*4, 3, 2, 1),                # 7 -> 4
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(feat*4, 1, 4, 1, 0)                      # 4 -> 1
        )

    def forward(self, x):
        return self.net(x).view(-1)  # (N,)

In [4]:
G = WGAN_Generator(z_dim=z_dim, feat=feat).to(device)
C = WGAN_Critic(feat=feat).to(device)

G.apply(dcgan_weights_init)
C.apply(dcgan_weights_init)

g_opt = optim.Adam(G.parameters(), lr=1e-4, betas=(0.0, 0.9))
c_opt = optim.Adam(C.parameters(), lr=1e-4, betas=(0.0, 0.9))

fixed_z = torch.randn(64, z_dim, 1, 1, device=device)

In [5]:
def gradient_penalty(C, real, fake, device):
    b = real.size(0)
    eps = torch.rand(b, 1, 1, 1, device=device)
    x_hat = eps * real + (1 - eps) * fake
    x_hat.requires_grad_(True)

    c_hat = C(x_hat)  # (b,)
    grads = torch.autograd.grad(
        outputs=c_hat,
        inputs=x_hat,
        grad_outputs=torch.ones_like(c_hat),
        create_graph=True,
        retain_graph=True,
        only_inputs=True
    )[0]

    grads = grads.view(b, -1)
    gp = ((grads.norm(2, dim=1) - 1) ** 2).mean()
    return gp

In [ ]:
epochs = 20
n_critic = 5
lambda_gp = 10.0

for epoch in range(1, epochs + 1):
    pbar = tqdm(train_loader, desc=f"WGAN-GP Epoch {epoch}/{epochs}")

    for real, _ in pbar:
        real = real.to(device, non_blocking=True)
        b = real.size(0)

        # Train Critic n_critic times
        for _ in range(n_critic):
            z = torch.randn(b, z_dim, 1, 1, device=device)
            fake = G(z).detach()

            c_real = C(real)
            c_fake = C(fake)

            gp = gradient_penalty(C, real, fake, device)
            c_loss = -(c_real.mean() - c_fake.mean()) + lambda_gp * gp

            c_opt.zero_grad(set_to_none=True)
            c_loss.backward()
            c_opt.step()

        # Train Generator once
        z = torch.randn(b, z_dim, 1, 1, device=device)
        fake = G(z)
        g_loss = -C(fake).mean()

        g_opt.zero_grad(set_to_none=True)
        g_loss.backward()
        g_opt.step()

        pbar.set_postfix(c_loss=float(c_loss.item()), g_loss=float(g_loss.item()), gp=float(gp.item()))

    # Save samples + checkpoint
    G.eval()
    with torch.no_grad():
        samples = G(fixed_z)

    save_grid(samples, paths.samples_dir / f"epoch_{epoch:03d}.png", nrow=8)

    torch.save({
        "epoch": epoch,
        "G": G.state_dict(),
        "C": C.state_dict(),
        "z_dim": z_dim,
        "feat": feat,
        "n_critic": n_critic,
        "lambda_gp": lambda_gp
    }, paths.models_dir / f"ckpt_epoch_{epoch:03d}.pth")

WGAN-GP Epoch 20/20: 100%|██████████| 469/469 [00:29<00:00, 16.14it/s, c_loss=-1.15, g_loss=50.2, gp=0.00978]
